# Generation Perturbation Sanity Check

Verifies that the generation experiment infrastructure works correctly before running the full experiment.

**Hypothesis under test:** Does the assistant axis produce qualitatively distinct steering effects on text generation compared to other directions (random, PCA PC1, factual-creative contrast), or is activation variance along the steering direction the primary explanatory factor?

**Checks:**
1. Normal generation works and `output_scores` has correct shape
2. Persistent perturbation hook produces different text
3. One-shot hook fires exactly once
4. All directions are orthogonal (critical — ensures we compare independent directions)
5. Single prompt: baseline vs every direction (qualitative comparison)
6. Per-step metrics compute correctly

In [ ]:
import torch
import numpy as np
from pathlib import Path

from generation_experiment import (
    SteeringExperiment,
    MODEL_CONFIGS,
    DEFAULT_PROMPTS,
    make_perturbation,
    _PerturbationHook,
    _OneShotPerturbationHook,
    compute_directions,
    generate_baseline,
    generate_perturbed,
    compute_step_metrics,
)

In [ ]:
# ---- Configuration ----
MODEL_NAME = "Qwen/Qwen3-32B"
MAX_NEW_TOKENS = 64  # short for sanity check
TEST_ALPHA = 1.0

In [ ]:
# ---- Load model ----
exp = SteeringExperiment(MODEL_NAME, deterministic=True)
target_layer = MODEL_CONFIGS[MODEL_NAME]["target_layer"]
print(f"Model: {MODEL_NAME}")
print(f"Layers: {exp.num_layers}, Hidden dim: {exp.hidden_dim}")
print(f"Target layer: {target_layer}")

## Check 1: Normal Generation

In [ ]:
test_prompt = "What is the capital of France?"
input_ids = exp.tokenize(test_prompt)
prompt_len = input_ids.shape[1]

bl_ids, bl_scores = generate_baseline(exp, input_ids, max_new_tokens=MAX_NEW_TOKENS)
bl_text = exp.tokenizer.decode(bl_ids[0, prompt_len:], skip_special_tokens=True)

print(f"Prompt: {test_prompt}")
print(f"Prompt tokens: {prompt_len}")
print(f"Generated tokens: {bl_ids.shape[1] - prompt_len}")
print(f"Scores: {len(bl_scores)} steps, each shape {bl_scores[0].shape}")
print(f"\nGenerated text:\n{bl_text}")

# Verify determinism: generate again
bl_ids_2, _ = generate_baseline(exp, input_ids, max_new_tokens=MAX_NEW_TOKENS)
assert (bl_ids == bl_ids_2).all(), "FAIL: Baseline generation is not deterministic!"
print("\nDeterminism check: PASSED (two runs identical)")

## Check 2: Persistent Perturbation Hook

In [ ]:
# Get baseline activation for delta scaling
baseline_acts, _ = exp.get_baseline_trajectory(input_ids)
h_L = baseline_acts[target_layer]

# Build assistant-away direction
axis_dir = exp.axis[target_layer].float()
axis_dir = axis_dir / axis_dir.norm()
delta = make_perturbation(h_L, -axis_dir, alpha=TEST_ALPHA)

# Generate with persistent hook
pt_ids, pt_scores = generate_perturbed(
    exp, input_ids, target_layer, delta, "persistent",
    max_new_tokens=MAX_NEW_TOKENS
)
pt_text = exp.tokenizer.decode(pt_ids[0, prompt_len:], skip_special_tokens=True)

print(f"BASELINE:\n{bl_text[:300]}")
print(f"\nPERTURBED (persistent, away, alpha={TEST_ALPHA}):\n{pt_text[:300]}")

first_10_match = (bl_ids[0, prompt_len:prompt_len+10] == pt_ids[0, prompt_len:prompt_len+10]).all().item()
print(f"\nFirst 10 tokens match: {first_10_match}")
if not first_10_match:
    print("PASSED: Perturbation changes generation")
else:
    print("WARNING: First 10 tokens identical despite perturbation -- alpha may be too small")

## Check 3: One-Shot Hook Fires Once

In [ ]:
# Generate with one-shot hook
delta_device = delta.to(exp.model.dtype)
layer_module = exp.layers[target_layer]
hook = _OneShotPerturbationHook(layer_module, delta_device)

with hook:
    with torch.inference_mode():
        output = exp.model.generate(
            input_ids, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, output_scores=True,
            return_dict_in_generate=True,
        )

os_ids = output.sequences
os_text = exp.tokenizer.decode(os_ids[0, prompt_len:], skip_special_tokens=True)

print(f"Hook fired: {hook.fired}")
assert hook.fired, "FAIL: Hook did not fire!"
print("PASSED: Hook fired exactly once")

print(f"\nBASELINE:    {bl_text[:150]}")
print(f"ONE-SHOT:    {os_text[:150]}")
print(f"PERSISTENT:  {pt_text[:150]}")

# One-shot should differ from baseline (perturbation has effect via KV cache)
os_matches_bl = (os_ids[0, prompt_len:] == bl_ids[0, prompt_len:min(os_ids.shape[1], bl_ids.shape[1])]).all().item()
print(f"\nOne-shot matches baseline: {os_matches_bl}")
if not os_matches_bl:
    print("PASSED: One-shot perturbation affects generation")

## Check 4: Direction Orthogonality

In [ ]:
print("Computing all directions (this runs ~60 forward passes)...\n")
directions = compute_directions(
    exp, target_layer, n_random_dirs=1, seed=42,
    enable_assistant=True, enable_random=True,
    enable_fc=True, enable_pca=True,
)

print(f"\nDirection names: {list(directions.keys())}")
print(f"\nPairwise cosine similarities:")
names = list(directions.keys())
all_orthogonal = True
for i, n1 in enumerate(names):
    for n2 in names[i+1:]:
        # Skip pairs that are negatives of each other (away/toward, +/-)
        cos = torch.dot(directions[n1], directions[n2]).item()
        tag = ""
        if abs(cos) > 0.99:
            tag = " (expected: negated pair)"
        elif abs(cos) > 0.01:
            tag = " *** NOT ORTHOGONAL ***"
            all_orthogonal = False
        print(f"  cos({n1}, {n2}) = {cos:+.6f}{tag}")

if all_orthogonal:
    print("\nPASSED: All independent directions are orthogonal")
else:
    print("\nWARNING: Some directions are not orthogonal")

## Check 5: Baseline vs Every Direction

In [ ]:
prompt = "Explain quantum computing to a five-year-old."
input_ids = exp.tokenize(prompt)
prompt_len = input_ids.shape[1]

bl_ids, bl_scores = generate_baseline(exp, input_ids, max_new_tokens=MAX_NEW_TOKENS)
bl_text = exp.tokenizer.decode(bl_ids[0, prompt_len:], skip_special_tokens=True)
baseline_acts, _ = exp.get_baseline_trajectory(input_ids)
h_L = baseline_acts[target_layer]

print(f"Prompt: {prompt}\n")
print(f"BASELINE:\n{bl_text}\n")
print("=" * 70)

for dir_name, direction in directions.items():
    delta = make_perturbation(h_L, direction, alpha=TEST_ALPHA)
    pt_ids, pt_scores = generate_perturbed(
        exp, input_ids, target_layer, delta, "persistent",
        max_new_tokens=MAX_NEW_TOKENS
    )
    pt_text = exp.tokenizer.decode(pt_ids[0, prompt_len:], skip_special_tokens=True)
    print(f"\n{dir_name} (persistent, alpha={TEST_ALPHA}):\n{pt_text}")
    print("-" * 70)
    del pt_ids, pt_scores

## Check 6: Per-Step Metrics

In [ ]:
# Compute metrics for one condition
delta = make_perturbation(h_L, directions["assistant_away"], alpha=TEST_ALPHA)
pt_ids, pt_scores = generate_perturbed(
    exp, input_ids, target_layer, delta, "persistent",
    max_new_tokens=MAX_NEW_TOKENS
)

metrics = compute_step_metrics(
    bl_scores, pt_scores, bl_ids, pt_ids, exp.tokenizer, prompt_len
)

print(f"Computed {len(metrics)} step metrics")
print(f"Keys: {list(metrics[0].keys())}")
print(f"\nFirst 10 steps:")
print(f"{'step':>4}  {'KL':>8}  {'JSD':>8}  {'match':>5}  {'bl_rank':>7}  {'top5_J':>6}  {'cos':>6}  {'baseline':>12}  {'perturbed':>12}")
print("-" * 90)
for m in metrics[:10]:
    print(f"{m['step']:>4}  {m['kl_divergence']:>8.4f}  {m['jensen_shannon_divergence']:>8.4f}  "
          f"{str(m['token_match']):>5}  {m['baseline_token_rank_in_perturbed']:>7}  "
          f"{m['top5_jaccard']:>6.2f}  {m['logit_cosine_similarity']:>6.4f}  "
          f"{repr(m['baseline_token']):>12}  {repr(m['perturbed_token']):>12}")

# Summary stats
import numpy as np
kls = [m['kl_divergence'] for m in metrics]
matches = [m['token_match'] for m in metrics]
print(f"\nSummary:")
print(f"  Mean KL: {np.mean(kls):.4f} (std={np.std(kls):.4f})")
print(f"  Token match rate: {np.mean(matches):.1%}")
print(f"  Mean top5 Jaccard: {np.mean([m['top5_jaccard'] for m in metrics]):.3f}")
print(f"  Mean logit cosine: {np.mean([m['logit_cosine_similarity'] for m in metrics]):.4f}")

del pt_ids, pt_scores
print("\nPASSED: Metrics computed successfully")

## Summary

If all checks passed, the infrastructure is ready. Run the full experiment with:
```bash
bash run_experiment.sh
```

The experiment will compare steering effects across direction types (assistant axis, random, FC contrast, PCA PC1) to determine whether the assistant axis is qualitatively special or whether variance along the steering direction is the main factor.